In [ ]:
from pyspark.sql.functions import col, count, sum as spark_sum, to_date, format_number
from pyspark.sql import SparkSession

# Создаем SparkSession
spark = SparkSession.builder \
    .appName("WebLogsAnalysis") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

# Загружаем данные
logs_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("work/web_server_logs.csv")

print("╔" + "═"*56 + "╗")
print("║                  АНАЛИЗ WEB-ЛОГОВ                      ║")
print("╚" + "═"*56 + "╝")

print("\n" + "┌" + "─"*56 + "┐")
print("│ ЗАДАНИЕ 1: Топ-10 самых активных IP                    │")
print("└" + "─"*56 + "┘")

# Топ-10 IP
top_ips = logs_df.groupBy("ip") \
    .agg(count("*").alias("request_count")) \
    .orderBy(col("request_count").desc()) \
    .limit(10)

print("\nTop 10 active IP addresses:")
top_ips.show(truncate=False)

print("\n" + "┌" + "─"*56 + "┐")
print("│ ЗАДАНИЕ 2: Количество запросов по HTTP-методам         │")
print("└" + "─"*56 + "┘")

# По методам
method_counts = logs_df.groupBy("method") \
    .agg(count("*").alias("method_count")) \
    .orderBy("method")

print("\nRequest count by HTTP method:")
method_counts.show(truncate=False)

print("\n" + "┌" + "─"*56 + "┐")
print("│ ЗАДАНИЕ 3: Количество запросов с кодом 404             │")
print("└" + "─"*56 + "┘")

# 404 ошибки
not_found_count = logs_df.filter(col("response_code") == 404).count()
print(f"\nNumber of 404 response codes: {not_found_count}")
total = logs_df.count()
percent = (not_found_count / total) * 100
print(f"Percentage of 404 errors: {percent:.2f}%")

print("\n" + "┌" + "─"*56 + "┐")
print("│ ЗАДАНИЕ 4: Сумма размеров ответов по дням              │")
print("└" + "─"*56 + "┘")

# По дням
daily_sizes = logs_df.withColumn("date", to_date(col("timestamp"))) \
    .groupBy("date") \
    .agg(spark_sum("response_size").alias("total_response_size")) \
    .orderBy("date")

print("\nTotal response size by day:")
# Форматируем числа с разделителями тысяч
daily_sizes_formatted = daily_sizes.withColumn(
    "total_response_size", 
    format_number("total_response_size", 0)
)
daily_sizes_formatted.show(truncate=False)
